# Desafio 2


In [2]:
import boto3
import os
from datetime import datetime
from botocore.exceptions import ClientError

print("🚀 AWS MANAGER - Backup S3 + Monitoreo EC2")
print("=" * 70)
print(f"🕒 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# ===========================================
# CONFIGURACIÓN - EDITA AQUÍ
# ===========================================
REGION = 'us-west-2'
BUCKET_NAME = 'bucket-lfrf'
S3_PREFIX = 'backup/'
LOCAL_DIR = r'C:\Users\Felipe\Desktop\prueba_backup'  # ¡TU RUTA!

# Clientes AWS
s3_client = boto3.client('s3', region_name=REGION)
ec2_client = boto3.client('ec2', region_name=REGION)

# ===========================================
# FUNCIÓN 1: BACKUP S3
# ===========================================
def backup_s3():
    print("\n📦 BACKUP S3")
    print("-" * 40)
    
    # Crear archivos de prueba
    os.makedirs(LOCAL_DIR, exist_ok=True)
    test_files = [
        ('doc1.txt', 'Backup prueba 1 - 18/03/2026'),
        ('doc2.txt', 'Backup prueba 2 - 18/03/2026'),
        ('datos.csv', 'col1,col2\nA,B\nC,D')
    ]
    
    for filename, content in test_files:
        path = os.path.join(LOCAL_DIR, filename)
        with open(path, 'w') as f:
            f.write(content)
        print(f"✅ Creado: {filename}")
    
    # Verificar bucket
    try:
        s3_client.head_bucket(Bucket=BUCKET_NAME)
        print(f"✅ Bucket '{BUCKET_NAME}' OK")
    except ClientError:
        print(f"❌ Error bucket")
        return False
    
    # Subir
    hoy = datetime.now().strftime('%Y-%m-%d')
    fecha_prefix = f"{S3_PREFIX}{hoy}/"
    subidos = 0
    
    for root, dirs, files in os.walk(LOCAL_DIR):
        for file in files:
            local_path = os.path.join(root, file)
            relative_path = os.path.relpath(local_path, LOCAL_DIR).replace(os.sep, '/')
            s3_key = f"{fecha_prefix}{relative_path}"
            
            s3_client.upload_file(local_path, BUCKET_NAME, s3_key)
            print(f"✅ {file} → s3://{BUCKET_NAME}/{s3_key}")
            subidos += 1
    
    print(f"🎉 BACKUP COMPLETADO: {subidos} archivos")
    return True

# ===========================================
# FUNCIÓN 2: MONITOREO EC2 (SIN pandas)
# ===========================================
def monitoreo_ec2():
    print("\n☁️  MONITOREO EC2")
    print("-" * 40)
    print(f"{'ID':<20} {'Estado':<12} {'Tipo':<12} {'IP Pública'}")
    print("-" * 60)
    
    try:
        response = ec2_client.describe_instances()
        total = 0
        
        for reservation in response['Reservations']:
            for instance in reservation['Instances']:
                instance_id = instance['InstanceId']
                estado = instance['State']['Name']
                tipo = instance['InstanceType']
                ip_publica = instance.get('PublicIpAddress', 'N/A')
                
                print(f"{instance_id:<20} {estado:<12} {tipo:<12} {ip_publica}")
                total += 1
        
        if total == 0:
            print("❌ No hay instancias EC2")
        else:
            print(f"\n📊 Total: {total} instancias")
            
    except Exception as e:
        print(f"❌ Error EC2: {e}")

# ===========================================
# EJECUTAR
# ===========================================
backup_ok = backup_s3()
monitoreo_ec2()

print("\n" + "="*70)
print("✅ TODO COMPLETADO ✓" if backup_ok else "⚠️  Backup falló")
print(f"🕒 Fin: {datetime.now().strftime('%H:%M:%S')}")


🚀 AWS MANAGER - Backup S3 + Monitoreo EC2
🕒 2026-03-18 23:25:45

📦 BACKUP S3
----------------------------------------
✅ Creado: doc1.txt
✅ Creado: doc2.txt
✅ Creado: datos.csv
✅ Bucket 'bucket-lfrf' OK
✅ doc1.txt → s3://bucket-lfrf/backup/2026-03-18/doc1.txt
✅ datos.csv → s3://bucket-lfrf/backup/2026-03-18/datos.csv
✅ doc2.txt → s3://bucket-lfrf/backup/2026-03-18/doc2.txt
🎉 BACKUP COMPLETADO: 3 archivos

☁️  MONITOREO EC2
----------------------------------------
ID                   Estado       Tipo         IP Pública
------------------------------------------------------------
❌ No hay instancias EC2

✅ TODO COMPLETADO ✓
🕒 Fin: 23:25:47
